# NB 6.5 &mdash; Més enllà de classificar

**MP 5149** &mdash; Desenvolupament de components software per a sistemes d'aprenentatge automàtic

**UT6** &mdash; Xarxes neuronals convolucionals i introducció a la visió per computador

---
### D'on venim

Fins ara, cada imatge rebia **una sola resposta**: «gat». Però una foto d'un carrer té cotxes, persones i
semàfors, i sovint volem saber quants n'hi ha i on són.

### Què farem avui

Aquesta sessió és curta i per jugar: no entrenarem res. Provarem models ja entrenats, que fan tres feines més:
**detectar** objectes, **retallar-los** píxel a píxel i **trobar les articulacions** d'una persona. Són
CNN per dins, de la família **YOLO** (*You Only Look Once*), i responen en dècimes de segon.

| Feina | Què respon | Model |
|---|---|---|
| Classificar | «Hi ha un gat» | ResNet18, al NB 6.4 |
| **Detectar** | «Hi ha un gat, i és dins d'aquest requadre» | `yolo11n.pt` |
| **Segmentar** | «Hi ha un gat, i aquests píxels són el gat» | `yolo11n-seg.pt` |
| **Poses** | «Aquí hi ha una persona, i aquest és el seu colze» | `yolo11n-pose.pt` |

### Preguntes que has de saber respondre en acabar

1. Què fa el llindar de confiança, i quin preu té abaixar-lo?
2. Què passa amb allò que no és a les 80 classes que coneix el model?
3. Aquests models aguanten els canvis que vam provar al crash test?
4. Quins problemes planteja detectar i seguir persones?

## 0. Preparació

Aquests models són de la biblioteca **Ultralytics**, amb llicència AGPL-3.0: és de codi obert i, si fas un programa
que la faci servir i el distribueixes o l'ofereixes per internet, n'has de compartir el codi. Per a aquest ús docent, no és cap problema. Cal instal·lar-la.

> ⏱ A Colab tarda menys d'un minut.

In [ ]:
!pip install -q ultralytics

Ara carreguem les fotos de mostra i definim dues funcions per veure els resultats.

In [ ]:
import urllib.request
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from ultralytics import YOLO

URL_PHOTOS = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/desenvolupament-components-software/refs/heads/main/UT06-Xarxes_convolucionals/fotos/"
PHOTOS = ["carrer_neu.jpg", "dublin_1901.jpg", "cafe.png", "gata.png", "coet.jpg"]
for name in PHOTOS:
    urllib.request.urlretrieve(URL_PHOTOS + name, name)


def show_result(result, size=8):
    """Dibuixa la imatge amb tot el que ha trobat el model."""
    plt.figure(figsize=(size, size))
    plt.imshow(result.plot()[:, :, ::-1])      # result.plot() torna els colors en ordre BGR
    plt.axis("off")
    plt.show()


def count_found(result):
    """Retorna quants objectes de cada classe ha trobat el model."""
    return dict(Counter(result.names[int(number)] for number in result.boxes.cls))

## 1. Detectar

Carreguem el model i li passem una foto. La primera vegada baixa els pesos, uns 6 MB. El paràmetre `conf`
és el **llindar de confiança**: només es dibuixa allò de què n'està prou segur. La funció `predict`
retorna una llista amb un resultat per imatge.

In [ ]:
detector = YOLO("yolo11n.pt")
result = detector.predict("carrer_neu.jpg", conf=0.25, verbose=False)[0]
print(count_found(result))
show_result(result)

**5 semàfors, 3 persones i 2 cotxes**, cadascun amb el seu requadre i un número: la seva confiança. Aquest model
tarda dècimes de segon. Fixa't també en el que no ha trobat: hi ha més cotxes, al fons. **Per què no els
veu?** Aquí entra el llindar.

### Ara tu · Missió 1: el llindar

Quan una xarxa detecta, proposa molts requadres, i cadascun té una confiança. El llindar decideix quins es queden.

1. **Aposta.** Si abaixes `conf` de 0,25 a 0,05 en aquesta foto, quants cotxes creus que trobarà? I
   si la puges a 0,9?
2. **Experimenta.** Fes una taula amb `count_found` per a `conf` = 0,05, 0,25, 0,5 i 0,9, amb `carrer_neu.jpg` i
   `dublin_1901.jpg`. Dibuixa el resultat de 0,05: quins requadres són errors?
3. **Explica.** Què hi guanyes i què hi perds abaixant el llindar? Quin llindar posaries en un sistema
   que atura un tren si veu una persona a la via? I en un que envia una multa automàtica?

In [ ]:
# El teu codi aquí

## 2. Segmentar i trobar poses

Els altres dos models s'usen exactament igual.

**Segmentar:** a més del requadre, pinta quins píxels formen l'objecte.

In [ ]:
segmenter = YOLO("yolo11n-seg.pt")
result = segmenter.predict("cafe.png", conf=0.25, verbose=False)[0]
print(count_found(result))
show_result(result, size=6)

**Poses:** busca 17 punts del cos de cada persona (ulls, espatlles, colzes, genolls…) i els uneix. Aquesta foto és
del carrer Westmoreland de Dublín, cap a l'any 1901.

In [ ]:
pose_model = YOLO("yolo11n-pose.pt")
result = pose_model.predict("dublin_1901.jpg", conf=0.25, verbose=False)[0]
print(count_found(result))
show_result(result)

Ha trobat **8 persones**, en una fotografia de fa més de cent anys. Aquests models s'han entrenat amb fotos
modernes, però un abric, unes espatlles i un cap continuen sent el mateix.

> **Norma de privacitat.** En aquest notebook, les imatges amb persones són només les que et donem. Si vols
> provar fotos pròpies, fes-les només d'objectes. Una detecció de persones és una mena de tractament
> de dades personals, i no es pot fer amb gent que no ho sap.

### Ara tu · Missió 2: què no sap

Aquests models s'han entrenat amb **COCO**, un conjunt amb només 80 classes. Pots veure-les totes amb
`detector.names`.

1. **Aposta.** Mira la llista. Què creus que dirà el detector de `coet.jpg`? I el model de poses, si li ensenyes la
   gata (`gata.png`)?
2. **Experimenta.** Passa les dues fotos pels tres models. Prova també un llindar baix.
3. **Busca** una altra cosa que sigui a la llista i una que no. Si vols, puja una foto d'un objecte amb la icona de
   la carpeta de Colab i passa-la pel detector.
4. **Explica.** Què fa amb una cosa que no coneix: se'n calla, o s'inventa alguna cosa? I el model de poses,
   amb la gata? Quina diferència hi ha amb el que va fer ResNet18 al NB 6.4?

In [ ]:
# El teu codi aquí

## 3. A la paret, una altra vegada

Al NB 6.3 vas veure que una CNN es refia del que ha vist. **Aquests models també?** Amb `Image.open` i els
mètodes de PIL es pot canviar una foto abans de passar-la al model, perquè `predict` també accepta imatges:

```python
image = Image.open("dublin_1901.jpg")
turned = image.rotate(90, expand=True)     # la gira 90 graus
tiny = image.resize((96, 71))              # la fa molt petita
detector.predict(turned, conf=0.25, verbose=False)[0]
```

### Ara tu · Missió 3: el crash test de YOLO

1. **Aposta.** Amb la foto de Dublín girada 90 graus, quantes persones
   creus que trobarà? I cap per avall? I si la redueixes a 96 píxels d'ample?
2. **Experimenta.** Fes una taula amb el nombre de persones per a l'original i per a girs de 45, 90 i 180 graus,
   i per a amples de 480, 240 i 96 píxels, totes amb `conf=0.25`.
3. **Dibuixa** el que passa amb el gir de 180 graus.
4. **Explica.** Com ho arreglaries si la càmera d'un dron ho veu tot des de dalt? Què has après al NB 6.2 que
   hi serviria?

In [ ]:
# El teu codi aquí

## 4. Resum

| | Classificar | Detectar | Segmentar | Poses |
|---|---|---|---|---|
| **Quants objectes** | Un per imatge | Molts | Molts | Moltes persones |
| **Què en sap** | La classe | Classe i requadre | Classe i píxels | Persona i 17 punts |
| **Per a què** | Ordenar fotos | Comptar, vigilar | Editar, mesurar | Esport, gestos, animació |

### Les idees que t'has d'endur

1. **El llindar és una decisió de negoci,** no un detall tècnic: tria entre avisar de més o avisar de menys.
2. **Un model no surt de les seves classes.** Si no és a la llista, no ho veurà o s'ho inventarà.
3. **Una CNN continua sent una CNN:** per més impressionant que sigui, aguanta el que ha vist.

## Exercici final

**Exercici 4. El comptador.** Escriu una funció `count_objects(path, class_name, conf=0.25)` que digui quants objectes
d'una classe hi ha en una foto, i prova-la amb les fotos de mostra. Compara-la amb el que comptes tu a ull. Amb quin llindar
s'acosta més? Per què no sempre n'hi ha prou amb una sola foto per triar-lo?

## Per al debat de classe

1. **Vigilància.** Tot el que heu fet avui es pot fer amb una càmera de carrer i un portàtil. Hi ha ciutats que ho
   fan servir per comptar gent, o per identificar-la. On posaries la línia? A Europa, el Reglament d'IA limita alguns
   d'aquests usos.
2. **Consentiment.** Les persones de Dublín de 1901 no van autoritzar res, i les de les fotos amb què s'han entrenat
   aquests models, tampoc gaire. Passa alguna cosa?
3. **Errors desiguals.** Si un detector de persones falla més amb determinats colors de pell o de roba, qui hauria de
   comprovar-ho abans de vendre'l?